# OSCD CPU benchmark
Run all cells on a fresh Colab runtime. Threshold choices use only training cities; the official test split is scored once at the end.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/oscd'
REPO_URL = 'https://github.com/Aravind-Subramanian-16/satellite-change-detection.git'
REPO_DIR = '/content/repo'
OSCD_DIR = '/content/oscd'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
drive_dir = Path(DRIVE_DIR)
for weight in ('model.onnx', 'best_model.pt'):
    source = drive_dir / 'weights' / weight
    if not source.is_file():
        raise FileNotFoundError(f'Missing required weight: {source}')

!mkdir -p /content/oscd
!unzip -q -o "$DRIVE_DIR/archive (1).zip" -d /content/oscd
!unzip -q -o "$DRIVE_DIR/test_labels.zip" -d /content/oscd

if Path(REPO_DIR).is_dir():
    %cd /content/repo
    !git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/repo

!cp "$DRIVE_DIR/weights/model.onnx" /content/repo/model.onnx
!cp "$DRIVE_DIR/weights/best_model.pt" /content/repo/best_model.pt
!grep -viE "^(onnx==|numpy)" requirements.txt > /content/req_colab.txt
!python -m pip -q install -r /content/req_colab.txt
import subprocess, sys
pytest_run = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
pytest_output = pytest_run.stdout + pytest_run.stderr
print(pytest_output, end='')
pytest_summary_line = next((line.strip() for line in reversed(pytest_output.splitlines()) if any(token in line for token in (' passed', ' failed', ' error', ' skipped'))), 'No pytest summary line emitted')
Path(DRIVE_DIR, 'results').mkdir(parents=True, exist_ok=True)
Path(DRIVE_DIR, 'results', 'pytest_summary.txt').write_text(pytest_summary_line + '\n')


In [ ]:
from pathlib import Path
images = Path(OSCD_DIR) / 'images' / 'Onera Satellite Change Detection dataset - Images'
train = [city.strip() for city in (images / 'train.txt').read_text().replace('\n', ',').split(',') if city.strip()]
test = [city.strip() for city in (images / 'test.txt').read_text().replace('\n', ',').split(',') if city.strip()]
print(f'Train cities ({len(train)}): {train}')
print(f'Test cities ({len(test)}): {test}')
assert len(train) == 14 and len(test) == 10
for split, cities, labels in [('train', train, Path(OSCD_DIR) / 'train_labels' / 'Onera Satellite Change Detection dataset - Train Labels'), ('test', test, Path(OSCD_DIR) / 'Onera Satellite Change Detection dataset - Test Labels')]:
    for city in cities:
        for directory in (images / city / 'imgs_1_rect', images / city / 'imgs_2_rect'):
            assert directory.is_dir(), f'{split} {city}: missing bands at {directory}'
        label_tif = labels / city / 'cm' / f'{city}-cm.tif'
        assert label_tif.is_file(), f'{split} {city}: missing label GeoTIFF at {label_tif}'
print('All cities have both acquisitions and a label GeoTIFF.')


In [ ]:
%cd /content/repo
# Results are checkpointed under Drive after every city; rerun resumes completed rows.
!python benchmarks/oscd_eval.py --root /content/oscd --output "$DRIVE_DIR/results"
!cat "$DRIVE_DIR/results/oscd_results_v2.md"


In [ ]:
from datetime import date
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

repo_dir = Path(REPO_DIR)
drive_results = Path(DRIVE_DIR) / 'results'
publish_dir = repo_dir / 'benchmarks' / 'results'
publish_dir.mkdir(parents=True, exist_ok=True)
for filename in ('oscd_results_v2.json', 'oscd_results_v2.md'):
    source = drive_results / filename
    if not source.is_file():
        raise FileNotFoundError(f'Missing saved benchmark result: {source}')
    shutil.copy2(source, publish_dir / filename)
for figure in drive_results.iterdir():
    if figure.suffix.lower() in {'.png', '.jpg', '.jpeg', '.svg', '.pdf'} and figure.stat().st_size < 2 * 1024 * 1024:
        shutil.copy2(figure, publish_dir / figure.name)

results_json = publish_dir / 'oscd_results_v2.json'
saved_results = json.loads(results_json.read_text())
metadata_records = [row for row in saved_results if row.get('record_type') == 'run_metadata']
if len(metadata_records) != 1:
    raise ValueError(f'Expected exactly one saved run metadata record, found {len(metadata_records)}')
run_metadata = metadata_records[0]['metadata']
run_metadata['code_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo_dir, text=True).strip()
run_metadata['run_date'] = date.today().isoformat()
pytest_summary_path = drive_results / 'pytest_summary.txt'
if not pytest_summary_path.is_file():
    raise FileNotFoundError(f'Missing captured pytest summary: {pytest_summary_path}')
run_metadata['pytest_summary_line'] = pytest_summary_path.read_text().strip()
results_json.write_text(json.dumps(saved_results, indent=2) + '\n')
sys.path.insert(0, str(repo_dir / 'benchmarks'))
from oscd_run_summary import write_run_summary
summary_path = publish_dir / 'RUN_SUMMARY.md'
summary_text = write_run_summary(results_json, summary_path)
print('```text\n' + summary_text + '```')

github_token = None
git_env = None
safe_remote = None
try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None
if not github_token:
    print('Push skipped: copy the RUN_SUMMARY block above')
else:
    try:
        original_remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=repo_dir, text=True).strip()
        remote_parts = urlsplit(original_remote)
        safe_netloc = remote_parts.hostname or ''
        if remote_parts.port:
            safe_netloc += f':{remote_parts.port}'
        safe_remote = urlunsplit((remote_parts.scheme, safe_netloc, remote_parts.path, remote_parts.query, remote_parts.fragment))
        repo_parts = urlsplit(REPO_URL)
        token_netloc = f'x-access-token:{quote(github_token, safe="")}@{repo_parts.netloc}'
        token_remote = urlunsplit((repo_parts.scheme, token_netloc, repo_parts.path, repo_parts.query, repo_parts.fragment))
        git_env = os.environ.copy()
        git_env['GIT_CONFIG_COUNT'] = '1'
        git_env['GIT_CONFIG_KEY_0'] = 'remote.origin.url'
        git_env['GIT_CONFIG_VALUE_0'] = token_remote
        subprocess.run(['git', 'add', 'benchmarks/results'], cwd=repo_dir, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        subprocess.run(['git', 'commit', '-m', f'results: OSCD v2 run {run_metadata["run_date"]}'], cwd=repo_dir, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        subprocess.run(['git', 'push', 'origin', 'main'], cwd=repo_dir, env=git_env, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print('Results pushed to main.')
    except Exception:
        print('Push skipped: copy the RUN_SUMMARY block above')
    finally:
        if safe_remote:
            subprocess.run(['git', 'remote', 'set-url', 'origin', safe_remote], cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if git_env is not None:
            for key in ('GIT_CONFIG_COUNT', 'GIT_CONFIG_KEY_0', 'GIT_CONFIG_VALUE_0'):
                git_env.pop(key, None)
        github_token = None
        token_remote = None
